# 03 — Model Training
ResNet50 fine-tuning for Know Your Batik classification.

In [1]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import sys
from pathlib import Path

import torch
import yaml

# Make sure project root is on sys.path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import get_dataloaders
from src.model import get_model
from src.preprocessor import get_class_weights

with open(ROOT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

PROCESSED_PATH = ROOT / cfg['data']['processed_path']
NUM_CLASSES    = cfg['data']['num_classes']
BATCH_SIZE     = cfg['model']['batch_size']
EPOCHS         = cfg['model']['epochs']
LR             = cfg['model']['learning_rate']
WEIGHT_DECAY   = 0.0001
PATIENCE       = cfg['model']['early_stopping']['patience']
MODELS_DIR     = ROOT / 'models'
OUTPUTS_DIR    = ROOT / 'outputs'
MODELS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# num_workers=0 avoids Windows multiprocessing issues
train_loader, val_loader, test_loader = get_dataloaders(
    processed_path=PROCESSED_PATH,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

class_weights = get_class_weights(PROCESSED_PATH).to(device)
model = get_model(NUM_CLASSES).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Device          : {device}')
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

Class mapping saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\models\class_labels.pkl
Train: 1701  Val: 213  Test: 213
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\fmoch/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [01:02<00:00, 1.64MB/s]


Device          : cpu
Total params    : 24,571,484
Trainable params: 23,126,556


In [2]:
# ── Cell 2: Training Config ────────────────────────────────────────────────
import torch.nn as nn
from torch.cuda.amp import GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer  = AdamW(trainable_params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion  = nn.CrossEntropyLoss(weight=class_weights)
scaler     = GradScaler(enabled=device.type == 'cuda')

CHECKPOINT_PATH = MODELS_DIR / 'checkpoint_best.pth'
best_val_acc    = 0.0
start_epoch     = 0

if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    best_val_acc = ckpt['best_val_acc']
    start_epoch  = ckpt['epoch'] + 1
    print(f'Resumed from epoch {ckpt["epoch"]}  (best val acc {best_val_acc:.4f})')
else:
    print('No checkpoint found — training from scratch')

No checkpoint found — training from scratch


C:\Users\fmoch\AppData\Local\Temp\ipykernel_10020\1354792612.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler     = GradScaler(enabled=device.type == 'cuda')


In [3]:
# ── Cell 3: Training Loop ─────────────────────────────────────────────────
from src.trainer import train_one_epoch, validate

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
epochs_without_improvement = 0

for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device, scaler
    )
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0
        torch.save(
            {
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_val_acc': best_val_acc,
            },
            CHECKPOINT_PATH,
        )
    else:
        epochs_without_improvement += 1

    print(
        f'Epoch {epoch + 1:>3}/{EPOCHS} | '
        f'train_loss {train_loss:.4f}  acc {train_acc:.4f} | '
        f'val_loss {val_loss:.4f}  acc {val_acc:.4f}'
        + ('  ✓' if epochs_without_improvement == 0 else '')
    )

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping triggered after {PATIENCE} epochs without improvement.')
        break

print(f'\nTraining complete. Best val acc: {best_val_acc:.4f}')

c:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\src\trainer.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(enabled=use_amp):


Epoch   1/50 | train_loss 2.6884  acc 0.3839 | val_loss 1.4164  acc 0.7183  ✓
Epoch   2/50 | train_loss 1.1724  acc 0.7278 | val_loss 0.6118  acc 0.8216  ✓
Epoch   3/50 | train_loss 0.6229  acc 0.8489 | val_loss 0.5172  acc 0.8638  ✓
Epoch   4/50 | train_loss 0.4412  acc 0.8777 | val_loss 0.4927  acc 0.8873  ✓
Epoch   5/50 | train_loss 0.3252  acc 0.9101 | val_loss 0.5773  acc 0.8685
Epoch   6/50 | train_loss 0.2348  acc 0.9383 | val_loss 0.5302  acc 0.8638
Epoch   7/50 | train_loss 0.1881  acc 0.9442 | val_loss 0.4799  acc 0.8826
Epoch   8/50 | train_loss 0.1588  acc 0.9565 | val_loss 0.4389  acc 0.8826
Epoch   9/50 | train_loss 0.1229  acc 0.9653 | val_loss 0.6016  acc 0.8826
Epoch  10/50 | train_loss 0.1094  acc 0.9659 | val_loss 0.5089  acc 0.8873
Epoch  11/50 | train_loss 0.1254  acc 0.9612 | val_loss 0.3711  acc 0.8826
Epoch  12/50 | train_loss 0.0807  acc 0.9759 | val_loss 0.5398  acc 0.8826
Epoch  13/50 | train_loss 0.1042  acc 0.9706 | val_loss 0.3472  acc 0.9014  ✓
Epoch  14/

In [4]:
# ── Cell 4: Save Final Model ───────────────────────────────────────────────
import json

# Load best weights back into model
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} (val acc {ckpt["best_val_acc"]:.4f})')

# Save final PyTorch model
final_model_path = MODELS_DIR / 'batik_classifier.pth'
torch.save(model.state_dict(), final_model_path)
print(f'Model saved → {final_model_path}')

# Save training history
history_path = MODELS_DIR / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training history saved → {history_path}')

Loaded best checkpoint from epoch 15 (val acc 0.9108)
Model saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\models\batik_classifier.pth
Training history saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\models\training_history.json


In [5]:
# ── Cell 5: Training Curves ────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epochs_ran  = len(history['train_loss'])
epoch_range = range(1, epochs_ran + 1)

# Best epoch = index of maximum val_acc
best_epoch_idx = history['val_acc'].index(max(history['val_acc']))
best_epoch_num = best_epoch_idx + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epoch_range, history['train_loss'], label='Train Loss')
ax1.plot(epoch_range, history['val_loss'],   label='Val Loss')
ax1.axvline(best_epoch_num, color='gray', linestyle='--', label=f'Best epoch ({best_epoch_num})')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

# Accuracy
ax2.plot(epoch_range, history['train_acc'], label='Train Acc')
ax2.plot(epoch_range, history['val_acc'],   label='Val Acc')
ax2.axvline(best_epoch_num, color='gray', linestyle='--', label=f'Best epoch ({best_epoch_num})')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

fig.suptitle('Know Your Batik — Training Curves', fontsize=14)
fig.tight_layout()

curves_path = OUTPUTS_DIR / 'training_curves.png'
fig.savefig(curves_path, dpi=150)
plt.close(fig)
print(f'Training curves saved → {curves_path}')

Training curves saved → C:\Users\fmoch\OneDrive\Documents\Data Kuliah\project\intermediate project\know-your-batik\outputs\training_curves.png
